In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
import os,uuid
import cohere
import pymupdf
from qdrant_client import QdrantClient, models
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
QDRANT_URL = "http://localhost:6333"
COLLECTION = "hybrid_rag"
EMBED_MODEL = "embed-v4.0"
EMBED_DIM = 1536
DATA_DIR = "../data/pdf"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

In [12]:
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))
qdr = QdrantClient(url=QDRANT_URL)
llm = ChatOpenAI(model="gpt-4.1")
print("Ready ---> ", QDRANT_URL)

Ready --->  http://localhost:6333


In [13]:
docs= []
for fileName in os.listdir(DATA_DIR):
    path = f"{DATA_DIR}/{fileName}"
    loader = PyMuPDFLoader(path)
    docs+= loader.load()

In [14]:
len(docs)

19

In [15]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
splitted_docs = splitter.split_documents(docs)
chunks = []
for d in splitted_docs:
    chunks.append({
        "source": d.metadata.get("source"),
        "page": d.metadata.get("page"),
        "text": d.page_content
    })

In [ ]:
def create(COLLECTION_NAME):
    qdr.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=EMBED_DIM,
                distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )

def create_collection(COLLECTION_NAME, isReset=False):

    if qdr.collection_exists(COLLECTION_NAME):
        if isReset:
            qdr.delete_collection(COLLECTION_NAME)
            create(COLLECTION_NAME)
    else:
        create(COLLECTION_NAME)

create_collection(COLLECTION, True)


collections=[CollectionDescription(name='hybrid_rag')]


In [26]:
def embed_documents(texts):
    result = co.embed(
        model=EMBED_MODEL,
        input_type="search_document",
        texts = texts,
        embedding_types=["float"]
    )
    
    return [list(value) for value in result.embeddings.float_]



def index_chunks(chunks, batch = 64): ##### 100 - [64:128]
    for i in range(0, len(chunks), batch):
        parts = chunks[i : i + batch]
        texts = [ chunk.get("text") for chunk in parts ]
        partEmbed = embed_documents(texts)
        
        vectorPoints = []
        
        for index, vector in enumerate(partEmbed):
            point = models.PointStruct(
                id = str(uuid.uuid4()),
                vector = {
                    "dense":vector,
                    "bm25": models.Document(text=texts[index], model="Qdrant/bm25")
                },
                payload = parts[index]
            )
            vectorPoints.append(point)
        
        qdr.upsert(COLLECTION, points=vectorPoints)
    return qdr.count(COLLECTION).count

In [27]:
count= index_chunks(chunks)
print("Indexed Points: ", count)

Indexed Points:  58


In [28]:
def embed_query(query):
    result = co.embed(
            model=EMBED_MODEL,
            input_type="search_query",
            texts=[query],
            embedding_types=["float"]
        )
    return list(result.embeddings.float_)[0]


def dense_search(query, k = 5):
    res = qdr.query_points(COLLECTION, query=embed_query(query), using="dense", limit=k, with_payload=True).points
    return res
    
    
def sparse_search(query, k = 5):
    res = qdr.query_points(COLLECTION, query=models.Document(text=query, model="Qdrant/bm25"), using="bm25", limit=k, with_payload=True).points
    return res